# Testing API requests

So far, we have used the basic ".../api/states/all" request without any additional tuning or optimization. Furthermore, we explored whether it is possible to schedule API requests automatically. At this stage, we are entering a broader decision-making process regarding the future architecture of the pipeline.<br>
A week ago, the initial idea was to run a Python script on a Raspberry Pi that would periodically fetch data from the API, store it locally, and rely on daily scheduled tasks orchestrated by the host operating system to manage the ETL pipeline.<br>Over the past few days, however, the concept evolved into building a dedicated home server running a Proxmox setup capable of hosting multiple virtual machines. The current stattus is that a  PostgreSQL database on the home server is setup and tested, allowing incoming API data to be injected directly into the database while Apache Airflow handles orchestration tasks from the host operating system.<br>In the near future, the complete Apache-based infrastructure will be migrated to the home server, which is designed for reliable 24/7 operation. As a result, the previously planned standalone Python script for data fetching will no longer be necessary. Overall, this architecture is significantly more production-oriented, scalable, and maintainable.<Hooray!>

## Extend the base API request, by additional parameters and OAuth2

Before testing scheduled API requests, we first need to explore and evaluate how to perform precise API calls and retrieve the desired data efficiently. Once we better understand the boundaries and capabilities of the API, we can further refine the scheduling intervals, define suitable time windows, and optimize the scope and structure of the requested data.

### Longitude and Latitude

In [11]:
import requests
import pandas as pd
# =========================
# Import paths and working directory
# import plugins from project
# =========================

import sys
sys.path.append(r"D:/aviation_data_pipeline")
import os
os.chdir(r"D:\aviation_data_pipeline")

from plugins.etl.transform import transform_raw_data


# =========================
# URLS (EDIT HERE IF NEEDED)
# =========================
BASE_URL = "https://opensky-network.org/api/states/all"

LA_MIN = 45.8389
LO_MIN = 5.9962
LA_MAX = 47.8229
LO_MAX = 10.5226

URL_FULL = BASE_URL
URL_FILTERED = f"{BASE_URL}?lamin={LA_MIN}&lomin={LO_MIN}&lamax={LA_MAX}&lomax={LO_MAX}"

To ensure a smooth workflow when working across notebooks and dataframes, we import reusable modules directly from the project’s plugin structure. This approach allows us to leverage existing, tested functionality without duplicating code. As a result, intermediate steps no longer need to be copied or rewritten inside notebooks or infrastructure components, improving consistency, maintainability, and overall development efficiency.

In [4]:
response_full = requests.get(URL_FULL)
data_full = response_full.json()

In [5]:
data_full_df = transform_raw_data(data_full)

In [6]:
data_full_df

,time,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1778408891,39de4e,TVF18NF,France,1.778409e+09,1778408890,8.8770,45.3026,11574.78,False,226.90,305.96,0.33,None,11711.94,5516,False,0
1,1778408891,80162c,AXB1507,India,1.778409e+09,1778408890,76.8446,26.8738,11277.60,False,258.40,178.06,0.33,None,11879.58,None,False,0
2,1778408891,e8027c,LPE2405,Chile,1.778409e+09,1778408776,-43.7021,-23.2485,6164.58,False,193.10,237.62,2.60,None,6537.96,None,False,0
3,1778408891,801638,AXB2817,India,1.778409e+09,1778408840,77.6433,13.2075,1173.48,False,69.45,90.42,-3.58,None,1287.78,None,False,0
4,1778408891,39de4b,TVF12ZW,France,1.778409e+09,1778408890,26.1087,41.8985,9418.32,False,259.79,115.45,-5.53,None,9700.26,0667,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6629,1778408891,7114c6,SVA263,Saudi Arabia,1.778409e+09,1778408604,31.6776,38.0583,10363.20,False,213.10,319.50,0.00,None,10744.20,4267,False,0
6630,1778408891,3b7ba7,DRAGOD,France,1.778409e+09,1778408883,-0.3180,49.2200,304.80,False,71.20,217.37,-0.33,None,342.90,None,False,0
6631,1778408891,abc0e4,SWA1912,United States,1.778409e+09,1778408891,-83.4793,27.9545,8915.40,False,239.41,111.16,-10.73,None,9410.70,2162,False,0
6632,1778408891,458664,CAT302,Denmark,1.778409e+09,1778408891,21.7719,41.6490,11582.40,False,222.44,340.27,0.00,None,11871.96,2024,False,0


In [12]:
response_filtered = requests.get(URL_FILTERED)
data_filtered = response_filtered.json()
data_filtered_df = transform_raw_data(data_filtered)

In [13]:

data_filtered_df

,time,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1778409150,39de59,TVF701,France,1778409149,1778409150,7.6862,45.9459,11582.40,False,215.45,303.31,0.00,None,11711.94,1000,False,0
1,1778409150,4bc8d3,PGT7039,Turkey,1778409150,1778409150,10.2607,47.7441,8770.62,False,205.96,267.57,-12.35,None,8961.12,6005,False,0
2,1778409150,4acb45,SAS841,Sweden,1778409149,1778409150,8.4593,47.7755,2628.90,False,133.28,191.35,-7.80,None,2705.10,0735,False,0
3,1778409150,46b82a,AEE616,Greece,1778409149,1778409150,6.0100,46.7286,9144.00,False,247.07,294.22,0.00,None,9296.40,1000,False,0
4,1778409150,8960f7,UAE25K,United Arab Emirates,1778409149,1778409149,9.3998,47.4575,11887.20,False,251.63,101.68,-0.33,None,12009.12,7562,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,1778409150,4b17f8,SWR8YE,Switzerland,1778409145,1778409145,8.5656,47.4552,NaN,True,0.00,132.19,NaN,None,NaN,1000,False,0
111,1778409150,4b17f9,SWR3BP,Switzerland,1778409150,1778409150,8.8765,47.1846,5242.56,False,167.01,131.00,5.85,None,5356.86,1000,False,0
112,1778409150,4b17e6,SWR2LB,Switzerland,1778409149,1778409149,8.5584,47.4487,NaN,True,0.00,244.69,NaN,None,NaN,3056,False,0
113,1778409150,4b17fd,SWR8WA,Switzerland,1778409139,1778409146,8.5667,47.4551,457.20,True,0.00,132.19,NaN,None,NaN,1000,False,0


Are the results within the desired values ?

In [21]:
data_filtered_df[
    (data_filtered_df["longitude"] > LO_MIN) &
    (data_filtered_df["longitude"] < LO_MAX) &
    (data_filtered_df["latitude"] > LA_MIN) &
    (data_filtered_df["latitude"] < LA_MAX)
].shape

(115, 18)

Looks promising.